# ความท้าทาย: การวิเคราะห์ข้อความเกี่ยวกับวิทยาศาสตร์ข้อมูล

ในตัวอย่างนี้ มาออกกำลังกายง่ายๆ ที่ครอบคลุมทุกขั้นตอนของกระบวนการวิทยาศาสตร์ข้อมูลแบบดั้งเดิม คุณไม่จำเป็นต้องเขียนโค้ดใดๆ สามารถคลิกที่เซลล์ด้านล่างเพื่อดำเนินการและสังเกตผลลัพธ์ได้เลย ในฐานะความท้าทาย ขอแนะนำให้คุณลองใช้โค้ดนี้กับข้อมูลที่แตกต่างกัน

## เป้าหมาย

ในบทเรียนนี้ เราได้พูดคุยเกี่ยวกับแนวคิดต่างๆ ที่เกี่ยวข้องกับวิทยาศาสตร์ข้อมูล มาให้ลองค้นพบแนวคิดที่เกี่ยวข้องเพิ่มเติมโดยการทำ **text mining** เราจะเริ่มต้นด้วยข้อความเกี่ยวกับวิทยาศาสตร์ข้อมูล ดึงคำสำคัญออกมา และจากนั้นลองสร้างภาพผลลัพธ์

สำหรับข้อความ ฉันจะใช้หน้าวิทยาศาสตร์ข้อมูลจากวิกิพีเดีย:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## ขั้นตอนที่ 1: การรับข้อมูล

ขั้นตอนแรกในกระบวนการวิทยาศาสตร์ข้อมูลทุกครั้งคือการรับข้อมูล เราจะใช้ไลบรารี `requests` เพื่อทำเช่นนั้น:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## ขั้นตอนที่ 2: การแปลงข้อมูล

ขั้นตอนถัดไปคือการแปลงข้อมูลให้อยู่ในรูปแบบที่เหมาะสมสำหรับการประมวลผล ในกรณีของเรา เราได้ดาวน์โหลดซอร์สโค้ด HTML จากหน้าเว็บ และเราจำเป็นต้องแปลงมันเป็นข้อความธรรมดา

มีหลายวิธีในการทำเช่นนี้ เราจะใช้ [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/) ไลบรารี Python ยอดนิยมสำหรับการแยกวิเคราะห์ HTML BeautifulSoup ช่วยให้เราสามารถเจาะจงเป้าหมายเป็นส่วนประกอบ HTML เฉพาะ เพื่อที่เราจะโฟกัสไปที่เนื้อหาหลักของบทความจากวิกิพีเดียและลดเมนูนำทาง, แถบด้านข้าง, ส่วนท้าย และเนื้อหาอื่นๆ ที่ไม่เกี่ยวข้อง (แม้ว่าบางข้อความบูทสแตรปยังอาจเหลืออยู่บ้าง)


ก่อนอื่น เราต้องติดตั้งไลบรารี BeautifulSoup สำหรับการแยกวิเคราะห์ HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## ขั้นตอนที่ 3: การได้มาซึ่งข้อมูลเชิงลึก

ขั้นตอนที่สำคัญที่สุดคือการเปลี่ยนข้อมูลของเราให้อยู่ในรูปแบบที่เราสามารถดึงข้อมูลเชิงลึกออกมาได้ ในกรณีของเรา เราต้องการสกัดคีย์เวิร์ดจากข้อความ และดูว่าคีย์เวิร์ดไหนมีความหมายมากกว่า

เราจะใช้ไลบรารี Python ที่เรียกว่า [RAKE](https://github.com/aneesha/RAKE) สำหรับการสกัดคีย์เวิร์ด ก่อนอื่น มาติดตั้งไลบรารีนี้ในกรณีที่ยังไม่มี: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

ฟังก์ชันหลักสามารถใช้งานได้จากออบเจ็กต์ `Rake` ซึ่งเราสามารถปรับแต่งได้โดยใช้พารามิเตอร์บางอย่าง ในกรณีของเรา เราจะตั้งค่าความยาวขั้นต่ำของคำสำคัญเป็น 5 ตัวอักษร ความถี่ขั้นต่ำของคำสำคัญในเอกสารเป็น 3 และจำนวนคำสูงสุดในคำสำคัญเป็น 2 สามารถลองปรับเปลี่ยนค่าอื่น ๆ และสังเกตผลลัพธ์ได้ตามใจชอบ


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


เราได้รับรายการคำศัพท์พร้อมกับระดับความสำคัญที่เกี่ยวข้อง ตามที่คุณเห็น สาขาที่เกี่ยวข้องมากที่สุด เช่น การเรียนรู้ของเครื่องและข้อมูลขนาดใหญ่ ปรากฏอยู่ในรายการในตำแหน่งต้น ๆ

## ขั้นตอนที่ 4: การแสดงผลลัพธ์

ผู้คนสามารถตีความข้อมูลได้ดีที่สุดในรูปแบบภาพ ดังนั้นจึงมักจะมีเหตุผลในการแสดงข้อมูลในรูปแบบภาพเพื่อดึงข้อมูลเชิงลึกบางอย่าง เราสามารถใช้ไลบรารี `matplotlib` ใน Python เพื่อวาดการแจกแจงง่ายๆ ของคำสำคัญพร้อมกับความเกี่ยวข้องของพวกมัน:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

อย่างไรก็ตาม ยังมีวิธีที่ดีกว่าในการแสดงความถี่ของคำ—โดยใช้ **Word Cloud** เราจะต้องติดตั้งไลบรารีอีกตัวหนึ่งเพื่อสร้าง word cloud จากรายชื่อคำสำคัญของเรา


In [ ]:
!{sys.executable} -m pip install wordcloud

อ็อบเจ็กต์ `WordCloud` มีหน้าที่รับข้อความต้นฉบับ หรือรายการคำที่คำนวณความถี่ไว้ล่วงหน้า และส่งคืนภาพ ซึ่งสามารถแสดงผลโดยใช้ `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

เรายังสามารถส่งข้อความต้นฉบับเข้าไปใน `WordCloud` ได้ด้วย - เรามาดูกันว่าเราจะได้ผลลัพธ์ที่คล้ายกันหรือไม่:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

คุณจะเห็นได้ว่า word cloud ตอนนี้ดูน่าประทับใจมากขึ้น แต่ก็ยังมีเสียงรบกวนมากมาย (เช่น คำที่ไม่เกี่ยวข้องอย่าง `Retrieved on`) นอกจากนี้ เรายังได้คีย์เวิร์ดที่มีสองคำลดลง เช่น *data scientist* หรือ *computer science* นั่นเป็นเพราะว่าอัลกอริทึม RAKE สามารถเลือกคีย์เวิร์ดที่ดีจากข้อความได้ดีกว่า ตัวอย่างนี้แสดงให้เห็นถึงความสำคัญของการประมวลผลและทำความสะอาดข้อมูลล่วงหน้า เพราะภาพที่ชัดเจนในตอนท้ายจะช่วยให้เราตัดสินใจได้ดีขึ้น

ในแบบฝึกหัดนี้ เราได้ผ่านกระบวนการง่ายๆ ในการสกัดความหมายบางอย่างจากข้อความในวิกิพีเดียในรูปแบบของคีย์เวิร์ดและ word cloud ตัวอย่างนี้ค่อนข้างง่าย แต่แสดงขั้นตอนทั่วไปทั้งหมดที่นักวิทยาศาสตร์ข้อมูลจะทำเมื่อทำงานกับข้อมูล ตั้งแต่การได้มาซึ่งข้อมูล ไปจนถึงการแสดงผลข้อมูล

ในหลักสูตรของเรา เราจะพูดถึงขั้นตอนเหล่านั้นทั้งหมดอย่างละเอียด


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
